In [ ]:
# Install pandas and numpy if not already installed
%pip install pandas numpy


In [ ]:
%pip install jupyter notebook

In [6]:
import pandas as pd
import numpy as np

## Let's talk about a key piece of the data munging process: Missing data.

The wine data we've been working with has been really nice :) No missing values, etc.

Let's move to another interesting data set, and start to make things a little harder.

In [37]:
##we'll import a fresh data set via a URL

data = pd.read_excel('https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls')

HTTPError: HTTP Error 502: Bad Gateway

In [ ]:
#display the first few rows

data.head()

Right away what do you notice? Check the Index against the first column. Seems a little duplicative.

In [ ]:
#re-import the data and use the first column as an Index, or just re-index.
#data.set_index(keys='Unnamed: 0', inplace=True) either will work
#data.set_index(keys='LIMIT_BAL: 0', inplace=True)

In [ ]:
#show the column names
data.columns

In [ ]:
#show the first few rows to confirm you've re-indexed correctly

data.head()

What else still seems weird?

In [ ]:
#assign the column names to the values in the first row. It's confusing to have two sets of column names

data.columns = data.iloc[0]

In [ ]:
#now drop the first row and get rid of it. now you should have a nice clean df. just a nice index on the left, and 
#column names that make sense up top.

data.drop(data.index[0], inplace=True)

In [ ]:
#take a peek at the data and confirm you've done this correctly
data.head()

Ok, now let's get some value counts. We want to build an understanding of how many null values we're dealing with here. Before we go any further though. Let's see what datatypes we're dealing with.

In [ ]:
data.dtypes

Everything was read as an Object. Please pause here and take moment to read online about why DataFrames infer the dtype of Object sometimes. This is an awesome article. If you read it carefully, you'll start to understand the various Pandas dtypes.

https://pbpython.com/pandas_dtypes.html

As a quick example, let's try to take some summary stats:

In [ ]:
#display summary stats for the dataframe

data.describe()

The summary stats you find above are probably not what you're expecting, right?

Seems like pandas is interpreting these variables as categorical instead of continuous. Let's fix that.

In [ ]:
data = data.apply(pd.to_numeric)

In [ ]:
#check the dtypes again and see if they've converted nicely

data.describe()

Looks like we now have a bunch of nice summary stats. Let's move on.

In [ ]:
#display the number of na values for each column

data.isna().sum()

Looks like we got lucky again :) Let's make things harder on ourselves and randomly apply NA values to this dataframe.

Let's do this using a mask for the dataframe.

In [ ]:
#create a masking array that is a random layout of 75% false values, and 25% true. feel free to do some reading
#on the various ways of doing this.

mask = np.random.random(data.shape) < 0.25

In [ ]:
#now, apply that mask to our data. let's create a new masked dataframe, instead of editing the old one:

data_with_nans = data.mask(mask)

In [ ]:
#confirm that the percentage of nans we added is ~ 25%

data_with_nans.isna().sum()/len(data)

In [ ]:
#show your newly created dataframe with the NaNs

data_with_nans.head()

Its worth mentioning here, that you'd almost never add NaNs to your data on purpose. We're just doing here to show what it's like to work with missing data.

In general, there are two basic ways of handling missing data:
 - Drop the rows with missing data. This is generally only the right answer if you wouldn't lose much of the dataset by doing so, and you think the rows containing NaNs are randomly distributed.
 
 
 - Impute some value to the NaNs. In most cases, you might consider imputing the average value of the column to the value that is missing, or depending on the nature of the data, you might impute something else (0, 1, True, False, mode, median, interpolated values from the preceding and following rows, forward filling, back filling, etc)
 
For our data above we know that about 25% of each column is NaN. This does NOT mean that if we dropped all NaNs we'd only drop 25% of rows. Let's figure out what we'd be left with if we dropped every row that doesn't contain full data.

In [ ]:
#this is trivial to do in Pandas. Drop the rows with NaN values below.

non_nans = data_with_nans.dropna()

In [ ]:
#print a statement that shows how many rows we started with and how many we're left with after dropping NaNs

print('The df had {} rows, but we we dropped all rows containing NaNs, \
it was reduced to {} rows.'.format(len(data_with_nans), len(non_nans)))

It should be very apparent that dropping all rows containing NaNs isn't a good option here. Instead let's move on and replace each NaN with the average value for it's row.

In [ ]:
#show the the means of each column:

data_with_nans.mean()

In [ ]:
filled_with_mean = data_with_nans.fillna(data_with_nans.mean())

In [ ]:
#Show the first few rows of the new df. Confirm the values that were NaN are now filled with the mean for the column.

filled_with_mean.head()

What do you notice about the above? You should notice some shortcomings of the fill method we used. There are tradeoffs between each fill method, but the probem above should be obvious.

What do you notice?

If you don't see it right away, take a look back at the definitions of each variable:
https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients

Answer: We've imputed averages to categorical variables. For example, in column two, 1=Male and 2=Female. So, imputing the average for that column isn't particularly helpful. 

## Let's save our dataframe with the NaN values. In the next lab we'll continue working on how to handle missing data.

Instead of saving it as a csv, let's introduce another cool library in Python. Pickle is used to save various objects to disk, for later use. Pickle is a popular way of saving models, data, etc.

In [ ]:
#import the pickle library

import pickle

In [ ]:
#save your the df you added the nans to as a .pickle file

data_with_nans.to_pickle('credit_data_with_nans.pickle')

In [ ]:
#read the pickle file back in as a dataframe, to confirm you've saved it correctly

df = pd.read_pickle('credit_data_with_nans.pickle')